In [1]:
project_root_path = '${PROJECT_ROOT_PATH}'
# @launchit.disable
project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]
# @launchit.stop

import sys
import os
import json
import multiprocessing as mp

import optuna
from optuna.trial import TrialState
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data
from torchvision import datasets
from torchvision import transforms

sys.path.append(project_root_path)
import lib.launchit # @launchit.disable
import lib.module_launcher # @launchit.disable

In [2]:
# @launchit.disable
with open(get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as cf:
    notebook_fname = json.load(cf)['jupyter_session']
    notebook_basename = os.path.basename(notebook_fname)
    notebook_name, notebook_ext = os.path.splitext(notebook_basename)

notebook_fname

'/home/misha/dev/mine/neurovision/experiment/optuna/optuna_torch_multiproc_test.ipynb'

In [3]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
BATCHSIZE = 128
CLASSES = 10
DIR = os.path.join(project_root_path, 'data', 'mnist')
EPOCHS = 10
N_TRAIN_EXAMPLES = BATCHSIZE * 30
N_VALID_EXAMPLES = BATCHSIZE * 10
STUDY_NAME = 'optuna_torch_miltiproc_test'

RUN_DIR = os.path.join(project_root_path, 'run', 'launchit', 'experiment', 'optuna')
os.makedirs(RUN_DIR, exist_ok=True)

STUDY_FNAME = os.path.join(RUN_DIR, STUDY_NAME + '.log')

In [4]:
def define_model(trial):
    # We optimize the number of layers, hidden units and dropout ratio in each layer.
    n_layers = trial.suggest_int("n_layers", 1, 3)
    layers = []

    in_features = 28 * 28
    for i in range(n_layers):
        out_features = trial.suggest_int("n_units_l{}".format(i), 4, 128)
        layers.append(nn.Linear(in_features, out_features))
        layers.append(nn.ReLU())
        p = trial.suggest_float("dropout_l{}".format(i), 0.2, 0.5)
        layers.append(nn.Dropout(p))

        in_features = out_features
    layers.append(nn.Linear(in_features, CLASSES))
    layers.append(nn.LogSoftmax(dim=1))

    return nn.Sequential(*layers)

In [5]:
def get_mnist():
    # Load FashionMNIST dataset.
    train_loader = torch.utils.data.DataLoader(
        datasets.FashionMNIST(DIR, train=True, download=True, transform=transforms.ToTensor()),
        batch_size=BATCHSIZE,
        shuffle=True,
    )
    valid_loader = torch.utils.data.DataLoader(
        datasets.FashionMNIST(DIR, train=False, transform=transforms.ToTensor()),
        batch_size=BATCHSIZE,
        shuffle=True,
    )

    return train_loader, valid_loader

In [6]:
def objective(trial):
    # Generate the model.
    model = define_model(trial).to(DEVICE)

    # Generate the optimizers.
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "RMSprop", "SGD"])
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    optimizer = getattr(optim, optimizer_name)(model.parameters(), lr=lr)

    # Get the FashionMNIST dataset.
    train_loader, valid_loader = get_mnist()

    # Training of the model.
    for epoch in range(EPOCHS):
        model.train()
        for batch_idx, (data, target) in enumerate(train_loader):
            # Limiting training data for faster epochs.
            if batch_idx * BATCHSIZE >= N_TRAIN_EXAMPLES:
                break

            data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)

            optimizer.zero_grad()
            output = model(data)
            loss = F.nll_loss(output, target)
            loss.backward()
            optimizer.step()

        # Validation of the model.
        model.eval()
        correct = 0
        with torch.no_grad():
            for batch_idx, (data, target) in enumerate(valid_loader):
                # Limiting validation data.
                if batch_idx * BATCHSIZE >= N_VALID_EXAMPLES:
                    break
                data, target = data.view(data.size(0), -1).to(DEVICE), target.to(DEVICE)
                output = model(data)
                # Get the index of the max log-probability.
                pred = output.argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()

        accuracy = correct / min(len(valid_loader.dataset), N_VALID_EXAMPLES)

        trial.report(accuracy, epoch)

        # Handle pruning based on the intermediate value.
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return accuracy

In [7]:
def run_optimization(params):
    print(f'params={params}')
    study = optuna.create_study(
        study_name=STUDY_NAME,
        direction='maximize',
        storage=JournalStorage(JournalFileBackend(file_path=STUDY_FNAME)),
        load_if_exists=True, # Useful for multi-process or multi-node optimization.
    )
    study.optimize(objective, n_trials=1, timeout=600)

In [8]:
# @launchit.disable
launch_fname = lib.launchit.launchit(notebook_fname, make_py_file=True, expandvars={'PROJECT_ROOT_PATH': project_root_path}, dir_name=RUN_DIR, verbosity=2)
mp_ctx = mp.get_context('spawn')

with mp_ctx.Pool(processes=4) as pool:
    lp_list = [lib.module_launcher.LaunchParameters(module_fname=launch_fname, func_name='run_optimization', func_args=i, verbosity=2) for i in range(10)]
    pool.map(lib.module_launcher.launch, lp_list)

# join and report
study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=JournalStorage(JournalFileBackend(file_path=STUDY_FNAME)),
    load_if_exists=True, # Useful for multi-process or multi-node optimization.
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

print("Study statistics: ")
print("  Number of finished trials: ", len(study.trials))
print("  Number of pruned trials: ", len(pruned_trials))
print("  Number of complete trials: ", len(complete_trials))

print("Best trial:")
trial = study.best_trial

print("  Value: ", trial.value)

print("  Params: ")
for key, value in trial.params.items():
    print("    {}: {}".format(key, value))

Creating /home/misha/dev/mine/neurovision/run/launchit/experiment/optuna/optuna_torch_multiproc_test-launch2.py
Created "/home/misha/dev/mine/neurovision/run/launchit/experiment/optuna/optuna_torch_multiproc_test-launch2.py"
Running optuna_torch_multiproc_test-launch2.run_optimization0
Running optuna_torch_multiproc_test-launch2.run_optimization1
Running optuna_torch_multiproc_test-launch2.run_optimization2
Running optuna_torch_multiproc_test-launch2.run_optimization3


[I 2026-01-23 17:12:53,735] A new study created in Journal with name: optuna_torch_miltiproc_test
[I 2026-01-23 17:12:53,769] Using an existing study with name 'optuna_torch_miltiproc_test' instead of creating a new one.
[I 2026-01-23 17:12:53,872] Using an existing study with name 'optuna_torch_miltiproc_test' instead of creating a new one.
[I 2026-01-23 17:12:53,955] Using an existing study with name 'optuna_torch_miltiproc_test' instead of creating a new one.
[I 2026-01-23 17:13:02,592] Trial 2 finished with value: 0.82890625 and parameters: {'n_layers': 1, 'n_units_l0': 85, 'dropout_l0': 0.20806233897372683, 'optimizer': 'Adam', 'lr': 0.007762115027651013}. Best is trial 2 with value: 0.82890625.
[I 2026-01-23 17:13:02,597] Using an existing study with name 'optuna_torch_miltiproc_test' instead of creating a new one.
[I 2026-01-23 17:13:02,599] Trial 3 finished with value: 0.26015625 and parameters: {'n_layers': 1, 'n_units_l0': 120, 'dropout_l0': 0.3577017877861176, 'optimizer': '

params=2
Finished optuna_torch_multiproc_test-launch2.run_optimization2, rv=None
Running optuna_torch_multiproc_test-launch2.run_optimization4
params=3
Finished optuna_torch_multiproc_test-launch2.run_optimization3, rv=None
Running optuna_torch_multiproc_test-launch2.run_optimization5


[I 2026-01-23 17:13:02,898] Trial 1 finished with value: 0.81796875 and parameters: {'n_layers': 2, 'n_units_l0': 31, 'dropout_l0': 0.3110987914057586, 'n_units_l1': 40, 'dropout_l1': 0.32446246611110885, 'optimizer': 'Adam', 'lr': 0.007494383265850468}. Best is trial 2 with value: 0.82890625.
[I 2026-01-23 17:13:02,903] Using an existing study with name 'optuna_torch_miltiproc_test' instead of creating a new one.


params=0
Finished optuna_torch_multiproc_test-launch2.run_optimization0, rv=None
Running optuna_torch_multiproc_test-launch2.run_optimization6


...

params=5
Finished optuna_torch_multiproc_test-launch2.run_optimization5, rv=None


[I 2026-01-23 17:13:11,292] Trial 8 pruned. 
[I 2026-01-23 17:13:11,320] Trial 6 finished with value: 0.76875 and parameters: {'n_layers': 1, 'n_units_l0': 115, 'dropout_l0': 0.48101443272001976, 'optimizer': 'RMSprop', 'lr': 0.01539331366038032}. Best is trial 2 with value: 0.82890625.
[I 2026-01-23 17:13:11,323] Trial 9 pruned. 


params=8
Finished optuna_torch_multiproc_test-launch2.run_optimization8, rv=None
params=6
Finished optuna_torch_multiproc_test-launch2.run_optimization6, rv=None
params=9
Finished optuna_torch_multiproc_test-launch2.run_optimization9, rv=None


[I 2026-01-23 17:13:11,608] Using an existing study with name 'optuna_torch_miltiproc_test' instead of creating a new one.


Study statistics: 
  Number of finished trials:  10
  Number of pruned trials:  3
  Number of complete trials:  7
Best trial:
  Value:  0.82890625
  Params: 
    n_layers: 1
    n_units_l0: 85
    dropout_l0: 0.20806233897372683
    optimizer: Adam
    lr: 0.007762115027651013
